# 01 - Data Loading & Exploratory Data Analysis (EDA)
## PhishScamSense: Real-Time Multimodal Phishing Defense

This notebook covers:
1. Loading phishing datasets from CSV and threat intelligence feeds
2. Basic dataset statistics and class distribution
3. URL length distribution analysis
4. Common patterns in phishing vs benign URLs

In [ ]:
import sys
import os

# Add project root to path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, PROJECT_ROOT)

import csv
import logging
from pathlib import Path
from collections import Counter
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print(f"Project root: {PROJECT_ROOT}")

## 1.1 Data Loading Functions

Functions to load data from local CSV and threat intelligence feeds (OpenPhish, PhishTank).

In [ ]:
import requests


def load_csv_dataset(filepath: str | Path) -> tuple[list[str], list[int]]:
    """Load URLs and labels from a CSV file (columns: url, label)."""
    urls, labels = [], []
    with open(filepath, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            urls.append(row["url"])
            labels.append(int(row["label"]))
    return urls, labels


def fetch_openphish_feed() -> list[str]:
    """Fetch latest phishing URLs from OpenPhish community feed."""
    try:
        response = requests.get("https://openphish.com/feed.txt", timeout=30)
        response.raise_for_status()
        urls = [line.strip() for line in response.text.splitlines() if line.strip()]
        logger.info(f"Fetched {len(urls)} URLs from OpenPhish")
        return urls
    except requests.RequestException as e:
        logger.error(f"Failed to fetch OpenPhish feed: {e}")
        return []


def fetch_phishtank_feed(api_key: str | None = None) -> list[str]:
    """Fetch latest phishing URLs from PhishTank."""
    try:
        url = "http://data.phishtank.com/data/online-valid.json"
        if api_key:
            url = f"http://data.phishtank.com/data/{api_key}/online-valid.json"
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        data = response.json()
        urls = [entry["url"] for entry in data]
        logger.info(f"Fetched {len(urls)} URLs from PhishTank")
        return urls
    except requests.RequestException as e:
        logger.error(f"Failed to fetch PhishTank feed: {e}")
        return []


print("Data loading functions defined.")

## 1.2 Load & Prepare Sample Dataset

We create a synthetic sample dataset to demonstrate the pipeline. Replace this with your actual dataset CSV in `data/raw/`.

In [ ]:
# Sample dataset for demonstration
# In production, load from: data/raw/phishing_dataset.csv
sample_urls = [
    # Benign URLs (label=0)
    "https://www.google.com/search?q=python",
    "https://github.com/anthropics/claude-code",
    "https://stackoverflow.com/questions/tagged/python",
    "https://en.wikipedia.org/wiki/Machine_learning",
    "https://www.youtube.com/watch?v=dQw4w9WgXcQ",
    "https://docs.python.org/3/library/urllib.html",
    "https://www.amazon.com/dp/B08N5WRWNW",
    "https://www.reddit.com/r/MachineLearning",
    "https://mail.google.com/mail/u/0/#inbox",
    "https://www.linkedin.com/in/johndoe",
    "https://www.microsoft.com/en-us/windows",
    "https://www.apple.com/macbook-pro",
    "https://www.netflix.com/browse",
    "https://twitter.com/home",
    "https://www.bbc.com/news/world",
    # Phishing URLs (label=1)
    "http://192.168.1.1/login/google-verify.html",
    "http://xn--ggle-1noa.com/accounts/login",
    "http://googl3-security.com/verify?user=admin&token=abc123",
    "http://paypa1-secure.com/signin/update-billing",
    "http://amaz0n-support.xyz/account/verify",
    "http://microsoft-365-login.tk/auth/signin",
    "http://netflix-billing-update.ml/payment",
    "http://faceb00k-security.ga/hacked/recovery",
    "http://apple-id-verify.cf/icloud/login.php",
    "http://bank0famerica-secure.ru/online/login",
    "http://dhl-tracking-update.info/parcel?id=83927492",
    "http://instagram-verify-account.net/auth",
    "http://linkedln-security.com/checkpoint/verify",
    "http://dropbox-shared-doc.tk/dl/invoice.pdf.exe",
    "http://wellsfarg0-alert.com/security/update",
]

sample_labels = [0]*15 + [1]*15

# Create DataFrame
df = pd.DataFrame({"url": sample_urls, "label": sample_labels})
df["label_name"] = df["label"].map({0: "benign", 1: "phishing"})

print(f"Dataset shape: {df.shape}")
print(f"\nClass distribution:")
print(df["label_name"].value_counts())
df.head()

## 1.3 Exploratory Data Analysis

In [ ]:
# Parse URL components
df["url_length"] = df["url"].str.len()
df["hostname"] = df["url"].apply(lambda u: urlparse(u).hostname or "")
df["scheme"] = df["url"].apply(lambda u: urlparse(u).scheme)
df["path"] = df["url"].apply(lambda u: urlparse(u).path)
df["num_dots"] = df["url"].str.count(r"\.")
df["num_hyphens"] = df["url"].str.count("-")
df["has_https"] = (df["scheme"] == "https").astype(int)
df["num_digits"] = df["url"].apply(lambda u: sum(c.isdigit() for c in u))

print("URL components extracted.")
df[["url", "hostname", "scheme", "url_length", "num_dots", "has_https", "label_name"]].head(10)

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
df["label_name"].value_counts().plot(kind="bar", ax=axes[0], color=["#2ecc71", "#e74c3c"])
axes[0].set_title("Class Distribution")
axes[0].set_ylabel("Count")
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

# Pie chart
df["label_name"].value_counts().plot(kind="pie", ax=axes[1], autopct="%1.1f%%",
                                      colors=["#2ecc71", "#e74c3c"])
axes[1].set_ylabel("")
axes[1].set_title("Class Proportion")

plt.tight_layout()
plt.show()

In [ ]:
# URL length distribution by class
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# URL length
for label, color in [(0, "#2ecc71"), (1, "#e74c3c")]:
    subset = df[df["label"] == label]
    name = "benign" if label == 0 else "phishing"
    axes[0].hist(subset["url_length"], bins=15, alpha=0.6, label=name, color=color)
axes[0].set_title("URL Length Distribution")
axes[0].set_xlabel("URL Length")
axes[0].legend()

# Number of dots
sns.boxplot(data=df, x="label_name", y="num_dots", ax=axes[1], palette=["#2ecc71", "#e74c3c"])
axes[1].set_title("Number of Dots in URL")

# Number of hyphens
sns.boxplot(data=df, x="label_name", y="num_hyphens", ax=axes[2], palette=["#2ecc71", "#e74c3c"])
axes[2].set_title("Number of Hyphens in URL")

plt.tight_layout()
plt.show()

In [ ]:
# HTTPS vs HTTP by class
fig, ax = plt.subplots(figsize=(8, 4))
https_counts = df.groupby(["label_name", "scheme"]).size().unstack(fill_value=0)
https_counts.plot(kind="bar", ax=ax, color=["#e74c3c", "#2ecc71"])
ax.set_title("HTTP vs HTTPS by Class")
ax.set_ylabel("Count")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

# TLD analysis
df["tld"] = df["hostname"].apply(lambda h: h.split(".")[-1] if "." in h else "")
print("\nTop TLDs by class:")
for label_name in ["benign", "phishing"]:
    subset = df[df["label_name"] == label_name]
    tld_counts = subset["tld"].value_counts().head(5)
    print(f"\n{label_name.upper()}:")
    print(tld_counts.to_string())

In [ ]:
# Summary statistics
print("=== Summary Statistics by Class ===\n")
numeric_cols = ["url_length", "num_dots", "num_hyphens", "num_digits", "has_https"]
summary = df.groupby("label_name")[numeric_cols].describe().T
print(summary)

# Save processed sample to CSV for use in other notebooks
DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
df.to_csv(os.path.join(DATA_DIR, "sample_dataset.csv"), index=False)
print(f"\nSample dataset saved to {DATA_DIR}/sample_dataset.csv")